# Run the Streamlit dashboard from Google Colab
Run the cells **in order** (Runtime -> Run all also works, but step 1 and 2 ask you to upload files).

1. Upload `customer_segmentation_project.zip` (it already contains the dataset and trained models).
2. *(Optional)* upload a different transaction file (CSV/Excel, same columns) to replace the dataset.
3. Cells install dependencies, check the saved models match Colab's scikit-learn version (retraining if needed, ~3-4 min), then start Streamlit.
4. The last cell prints a public `https://....trycloudflare.com` URL. Open it in a new tab.

**Keep the last cell running** - stopping it (or closing the Colab tab / letting the runtime idle out) shuts the app down. The URL changes every run, and free Colab runtimes are temporary (they disconnect after inactivity and after ~12 hours).

## Step 1 - Upload and unzip the project

In [ ]:
import os, sys, zipfile, pathlib
from google.colab import files
if not pathlib.Path('customer_segmentation_project/backend.py').exists():
    up = files.upload()                       # choose customer_segmentation_project.zip
    zipfile.ZipFile(next(iter(up))).extractall('.')
os.chdir('customer_segmentation_project')
print('Project folder:', os.getcwd()); print(sorted(os.listdir('.')))

## Step 2 (optional) - Use your own dataset
Skip this cell to use the included `data/online_retail_II.xlsx`. A new file must have columns `Invoice, StockCode, Quantity, InvoiceDate, Price, Customer ID, Country` (Description optional). Afterwards the models are retrained on it in Step 4.

In [ ]:
REPLACE_DATA = False          # <- set to True to upload a replacement dataset
DATA_FILE = 'data/online_retail_II.xlsx'
if REPLACE_DATA:
    up = files.upload(); name = next(iter(up))
    DATA_FILE = 'data/' + name; os.replace(name, DATA_FILE)
    for f in ['models/models.joblib']: 
        if os.path.exists(f): os.remove(f)          # force retraining
print('Dataset:', DATA_FILE)

## Step 3 - Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## Step 4 - Make sure the model files work in this Colab runtime
Pickled scikit-learn models are tied to the library version. If the saved models were built with a different version, they are rebuilt here from the dataset (takes ~3-4 minutes the first time).

In [ ]:
import importlib, subprocess, sklearn, joblib
sys.path.insert(0, os.getcwd())
import backend as B; importlib.reload(B)
need_train = not B.artifacts_exist()
if not need_train:
    meta = joblib.load('models/results.joblib')['meta']
    need_train = meta['sklearn_version'].split('.')[:2] != sklearn.__version__.split('.')[:2]
    print('saved with scikit-learn', meta['sklearn_version'], '| running', sklearn.__version__)
if need_train:
    print('Training models ...'); B.train_all(DATA_FILE)
else:
    print('Saved models are compatible - no retraining needed.')
art = B.load_artifacts(); print('Artifacts load OK:', list(art))

## Step 5 - Start Streamlit and open a public tunnel (Cloudflare)
Run this cell and **leave it running**. It prints the public URL when ready.

In [ ]:
import subprocess, time, re, urllib.request, os, signal
PORT = 8501
# stop anything left from a previous run
os.system("pkill -f 'streamlit run' ; pkill -f cloudflared")
st = subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', str(PORT), '--server.headless', 'true',
                       '--server.enableCORS', 'false', '--server.enableXsrfProtection', 'false', '--browser.gatherUsageStats', 'false'],
                      stdout=open('streamlit.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):                                   # wait until the app answers
    try: urllib.request.urlopen(f'http://localhost:{PORT}/_stcore/health', timeout=2); break
    except Exception: time.sleep(1)
else: raise RuntimeError('Streamlit did not start - see streamlit.log:\n' + open('streamlit.log').read()[-2000:])
print('Streamlit is running.')

if not os.path.exists('cloudflared'):
    os.system('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared')
cf = subprocess.Popen(['./cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}', '--no-autoupdate'],
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None; t0 = time.time(); log = []
while time.time() - t0 < 60 and url is None:
    line = cf.stdout.readline(); log.append(line)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m: url = m.group(0)
if url: print('\n>>> Open your dashboard here:', url, '\n(leave this cell running; stop it to shut the app down)')
else: print('No tunnel URL found. Cloudflare output:\n' + ''.join(log[-15:]) + '\nTry the LocalTunnel fallback below.')
try:
    while True: time.sleep(30)                       # keep-alive: the app lives as long as this cell runs
except KeyboardInterrupt:
    st.terminate(); cf.terminate(); print('Stopped.')

### Fallback: LocalTunnel (if Cloudflare is blocked)
Stop the cell above, then run:
```python
!npm install -q localtunnel
import subprocess, urllib.request
print('Tunnel password (IP):', urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode().strip())
!npx localtunnel --port 8501      # open the printed URL, paste the IP as the password
```
(Streamlit must already be running from the cell above - start it first without the tunnel part if needed.)

## Troubleshooting
| Symptom | Fix |
|---|---|
| `Streamlit did not start` | Read `streamlit.log` (printed in the error). Most often Step 3 was skipped. |
| Page says *Trained model artifacts were not found* | Run Step 4 (or click **Train models now** in the app). |
| Version warning in the sidebar | Re-run Step 4 - it retrains for this runtime. |
| Tunnel URL shows an error / 502 | Wait ~10 s and refresh; if it persists re-run Step 5 (a new URL is issued). |
| App stops after a while | Colab disconnected an idle runtime. Re-run Steps 1-5 (no need to re-upload if the files still exist). |
| Uploaded file rejected in the app | It needs columns `Invoice, StockCode, Quantity, InvoiceDate, Price, Customer ID, Country`; see the error text. |
| Out-of-memory when reading Excel | Use *Runtime -> Change runtime type -> High-RAM*, or convert the data to CSV first. |